# 11 LightGBM Interpretability and SHAP

This notebook builds the interpretability layer for the completed LightGBM rolling-origin analysis. It inspects feature use in the selected weather specifications, with M2 as the main operational weather model and M4 as a secondary uncertainty-extension check.

The notebook does not rerun rolling-origin evaluation, rerun the results audit, tune hyperparameters, or modify final prediction outputs. If saved rolling-origin estimators are unavailable, the helper script fits representative interpretability-only LightGBM models from the tuned parameter payload and feature registry. These models support feature-importance and SHAP diagnostics only; they are not new performance evidence.


## Guardrails

Inputs are read from the ML panel, feature registry, tuned parameter JSON and completed `full_step5` result artifacts. Smoke runs, notebook 07 benchmark outputs and notebook 09b alpha-sensitivity outputs are not used as result evidence.

SHAP and feature importance are model-behaviour diagnostics. They do not identify causal weather effects and do not replace the rolling-origin M2-vs-M1 performance comparison.


In [ ]:
import runpy
from pathlib import Path


def find_project_root() -> Path:
    candidates = [Path.cwd().resolve()]
    markers = ["README_FOR_CODEX.md", "docs/PROJECT_MAP.md", "data", "notebooks"]
    for start in candidates:
        for parent in [start] + list(start.parents):
            if all((parent / marker).exists() for marker in markers):
                return parent
    raise RuntimeError("Could not locate the project root from marker files.")


ROOT = find_project_root()
SCRIPT = ROOT / "tools" / "build_lightgbm_interpretability_shap.py"

print(f"Project root: {ROOT}")
print(f"Helper script: {SCRIPT}")
assert SCRIPT.exists(), "Missing interpretability helper script."

## Model-source configuration

The configuration prefers saved fitted models from notebook 09 when the `full_step5` model inventory is available. SHAP is computed for selected feature-set and horizon combinations to keep the interpretability layer tractable. If a saved model is unavailable for a selected combination, the helper falls back to a clearly labelled representative interpretability model.


In [ ]:
USE_SAVED_FULL_STEP5_MODELS_IF_AVAILABLE = True
SHAP_MODEL_SELECTION_MODE = "representative_from_saved"
SHAP_MAX_MODELS_PER_FEATURE_SET_HORIZON = 1
SHAP_SELECTED_FEATURE_SETS = ["M2", "M4"]
SHAP_SELECTED_HORIZONS_BY_FEATURE_SET = {
    "M2": [0, 1, 2, 3, 7, 10],
    "M4": [7, 10],
}
SHAP_SAVED_MODEL_ORIGIN_POLICY = "latest_origin"


## Build interpretability outputs

The helper script performs environment checks, validates feature sets, builds model-source summaries, computes LightGBM feature importance and SHAP summaries, and writes thesis-ready tables, figures, reports and documentation updates.


In [ ]:
runpy.run_path(
    str(SCRIPT),
    run_name="__main__",
    init_globals={
        "USE_SAVED_FULL_STEP5_MODELS_IF_AVAILABLE": USE_SAVED_FULL_STEP5_MODELS_IF_AVAILABLE,
        "SHAP_MODEL_SELECTION_MODE": SHAP_MODEL_SELECTION_MODE,
        "SHAP_MAX_MODELS_PER_FEATURE_SET_HORIZON": SHAP_MAX_MODELS_PER_FEATURE_SET_HORIZON,
        "SHAP_SELECTED_FEATURE_SETS": SHAP_SELECTED_FEATURE_SETS,
        "SHAP_SELECTED_HORIZONS_BY_FEATURE_SET": SHAP_SELECTED_HORIZONS_BY_FEATURE_SET,
        "SHAP_SAVED_MODEL_ORIGIN_POLICY": SHAP_SAVED_MODEL_ORIGIN_POLICY,
    },
)


## Main outputs

The main output folders are:

- `outputs/lightgbm_interpretability/tables/`
- `outputs/lightgbm_interpretability/figures/`
- `outputs/thesis_tables/interpretability/`

The main documentation outputs are:

- `docs/LIGHTGBM_INTERPRETABILITY_SHAP_REPORT.md` and `.html`
- `docs/SHAP_THESIS_INTERPRETATION_SUMMARY.md` and `.html`
- `docs/LIGHTGBM_INTERPRETABILITY_SHAP_CHANGELOG.md`

These outputs should be read as model-diagnostic evidence. Forecast-value conclusions remain anchored in the completed rolling-origin evaluation.
